In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()
print(f"saving figures:{SAVE_FIGURES}")

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal

NB_ID = "25"

In [ ]:
print("Loading Val Set...")
mixed_val_dir = get_unet_path(STAGE_MIXED, split=VAL)
print(f"Looking for data in: {mixed_val_dir}")

val_data = np.load(mixed_val_dir / f"{MIXED_DATASET}.npy")
val_df = pd.read_csv(mixed_val_dir / f"{MIXED_METADATA}.csv")

print(f"Tensor Shape: {val_data.shape}")
print(f"Metadata Rows: {len(val_df)}")
assert len(val_data) == len(val_df), "CRITICAL: Data and Metadata length mismatch!"

print("Loading Test Set...")
mixed_test_dir = get_unet_path(STAGE_MIXED, split=TEST)
print(f"Looking for data in: {mixed_test_dir}")

test_data = np.load(mixed_test_dir / f"{MIXED_DATASET}.npy")
test_df = pd.read_csv(mixed_test_dir / f"{MIXED_METADATA}.csv")

print(f"Tensor Shape: {test_data.shape}")
print(f"Metadata Rows: {len(test_df)}")
assert len(test_data) == len(test_df), "CRITICAL: Data and Metadata length mismatch!"

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Plot Validation
sns.histplot(val_df['sinr_db'], bins=len(val_df['sinr_db'].unique()), color='teal', discrete=True, ax=ax[0])
ax[0].set_title(f"Validation Set SINR Distribution")
ax[0].set_ylabel("Sample Count")
ax[0].set_xlabel("SINR (dB)")

# Plot Test
sns.histplot(test_df['sinr_db'], bins=len(test_df['sinr_db'].unique()), color='orange', discrete=True, ax=ax[1])
ax[1].set_title(f"Test Set SINR Distribution")
ax[1].set_ylabel("Sample Count")
ax[1].set_xlabel("SINR (dB)")

save_plot("val_test_sinr_distribution", machine_id="B", nb_id=NB_ID, fig_id="01")
plt.show()

In [ ]:
def plot_grid_heatmap(df, title, ax):
    # Cross-tabulate SINR vs Interference Type
    grid = pd.crosstab(df['sinr_db'], df['interference_type'])
    
    # Plot
    sns.heatmap(grid, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax, linewidths=0.5)
    ax.set_title(title)
    ax.set_ylabel("SINR (dB)")
    ax.set_xlabel("Interference Type")
    
    # Validation Logic
    if (grid.values == 15).all():
        return True
    return False

fig, ax = plt.subplots(1, 2, figsize=(18, 8))

# Validation Heatmap
val_pass = plot_grid_heatmap(val_df, "Validation Grid (Target: 15)", ax[0])
ax[0].add_patch(plt.Rectangle((0,0), 1, 1, fill=False, edgecolor='green', lw=0))

# Test Heatmap
test_pass = plot_grid_heatmap(test_df, "Test Grid (Target: 15)", ax[1])

plt.tight_layout()
save_plot("val_test_interference_grid", machine_id="B", nb_id=NB_ID, fig_id="02")
plt.show()

print(f"Validation Grid Status: {'PASS' if val_pass else 'FAIL'}")
print(f"Test Grid Status:       {'PASS' if test_pass else 'FAIL'}")

In [ ]:
# Select a specific clean file (ID 0) and specific interference (Spot)
# We pull this from the Validation set
subset = val_df[ (val_df['clean_frame_id'] == 0) & (val_df['interference_type'] == 'Spot') ]

# Define 3 target levels to visualize
targets = [-10.0, 4.0, 20.0]
labels = ["Hard (-10 dB)", "Medium (+4 dB)", "Easy (+20 dB)"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, target, label in zip(axes, targets, labels):
    # Find the row closest to the target SINR
    row = subset.iloc[(subset['sinr_db'] - target).abs().argsort()[:1]]
    idx = row.index[0]
    actual_sinr = row['sinr_db'].values[0]
    
    # Get Data
    sig = val_data[idx]
    
    # Plot
    ax.plot(sig[0, :500], label='In-Phase', alpha=0.9, linewidth=1.5)
    ax.plot(sig[1, :500], label='Quadrature', alpha=0.7, linewidth=1.5)
    
    ax.set_title(f"{label}")
    ax.set_ylim(-6, 6) 
    ax.grid(True, alpha=0.3)
    if target == -10: ax.legend(loc='upper right')

plt.suptitle("Spot Jamming Evolution (Clean File #0)")
plt.tight_layout()
save_plot("spot_jamming_evolution_clean", machine_id="B", nb_id=NB_ID, fig_id="03")
plt.show()

In [ ]:
# Same logic, but for Digital Interference
subset_dig = val_df[ (val_df['clean_frame_id'] == 5) & (val_df['interference_type'] == 'Digital3') ]

targets = [-10.0, 20.0]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, targets):
    row = subset_dig.iloc[(subset_dig['sinr_db'] - target).abs().argsort()[:1]]
    idx = row.index[0]
    actual_sinr = row['sinr_db'].values[0]
    
    sig = val_data[idx]
    
    ax.plot(sig[0, :500], alpha=0.9)
    ax.plot(sig[1, :500], alpha=0.7)
    ax.set_title(f"Digital Jamming {actual_sinr} dB")
    ax.set_ylim(-6, 6)
    ax.grid(True, alpha=0.3)

plt.suptitle("Digital Signal Overlap (Clean File #5)")
save_plot("digital_jamming_overlap_clean", machine_id="B", nb_id=NB_ID, fig_id="04")
plt.show()